In [ ]:
import numpy as np
import pandas as pd

# Set random seed untuk reproduksibilitas
np.random.seed(42)

# 1. GENERATE DIM_PRODUCT (Master Produk)
products_data = [
    {
        "SKU_Code": "SKU-001",
        "SKU_Name": "GlowCare Shampoo 170ml",
        "Category": "Personal Care",
        "Brand": "GlowCare",
        "Standard_COGS": 12000,
        "Base_List_Price": 22000,
    },
    {
        "SKU_Code": "SKU-002",
        "SKU_Name": "GlowCare Body Wash 250ml",
        "Category": "Personal Care",
        "Brand": "GlowCare",
        "Standard_COGS": 15000,
        "Base_List_Price": 28000,
    },
    {
        "SKU_Code": "SKU-003",
        "SKU_Name": "CleanMax Laundry Liquid 800ml",
        "Category": "Home Care",
        "Brand": "CleanMax",
        "Standard_COGS": 18000,
        "Base_List_Price": 35000,
    },
    {
        "SKU_Code": "SKU-004",
        "SKU_Name": "NutriCrunch Snack 100g",
        "Category": "Foods & Beverage",
        "Brand": "NutriCrunch",
        "Standard_COGS": 4500,
        "Base_List_Price": 9000,
    },
    {
        "SKU_Code": "SKU-005",
        "SKU_Name": "RefreshTea Jasmine 500ml",
        "Category": "Foods & Beverage",
        "Brand": "RefreshTea",
        "Standard_COGS": 2500,
        "Base_List_Price": 6000,
    },
]

df_dim_product = pd.DataFrame(products_data)

# 2. GENERATE FACT_SALES_MONTHLY
months = pd.date_range(start="2025-01-01", end="2025-12-01", freq="MS")
channels = ["General Trade", "Modern Trade", "E-Commerce"]

sales_rows = []

for m in months:
    month_str = m.strftime("%Y-%m")
    month_num = m.month

    for _, prod in df_dim_product.iterrows():
        sku = prod["SKU_Code"]
        base_price = prod["Base_List_Price"]
        std_cogs = prod["Standard_COGS"]

        for ch in channels:
            # Base Volume Penjualan berdasarkan Channel
            if ch == "General Trade":
                base_units = np.random.randint(12000, 18000)
            elif ch == "Modern Trade":
                base_units = np.random.randint(8000, 13000)
            else:
                base_units = np.random.randint(3000, 6000)

            # Musiman : Ramadan/Lebaran (Bulan 3-4) & Akhir Tahun (Bulan 12)
            if month_num in [3, 4, 12]:
                units_sold = int(base_units * np.random.uniform(1.25, 1.45))
            else:
                units_sold = int(base_units * np.random.uniform(0.9, 1.1))

            # Gross Revenue
            gross_revenue = units_sold * base_price

            # Diskonto Promo (Trade Discount)
            # ANOMALI 1: Pembengkakan Diskon di Modern Trade pada Q4 (Bulan 10-12)
            if ch == "Modern Trade" and month_num in [10, 11, 12]:
                trade_discount_pct = np.random.uniform(0.22, 0.28)  # High discount leakage
            elif ch == "Modern Trade":
                trade_discount_pct = np.random.uniform(0.12, 0.16)
            elif ch == "General Trade":
                trade_discount_pct = np.random.uniform(0.05, 0.08)
            else:
                trade_discount_pct = np.random.uniform(0.10, 0.15)

            trade_discount = round(gross_revenue * trade_discount_pct)

            # Rebates / Listing Fee (Off-Invoice)
            rebates_pct = np.random.uniform(0.02, 0.04) if ch == "Modern Trade" else np.random.uniform(0.005, 0.015)
            rebates_listing_fee = round(gross_revenue * rebates_pct)

            # Net Revenue
            net_revenue = gross_revenue - trade_discount - rebates_listing_fee

            # Actual COGS per Unit
            # ANOMALI 2: Cost Push / Margin Squeeze di Q3 (Bulan 7-9) untuk Personal Care
            if month_num in [7, 8, 9] and prod["Category"] == "Personal Care":
                actual_cogs_unit = std_cogs * np.random.uniform(1.18, 1.25)  # Kenaikan bahan baku 18-25%
            else:
                actual_cogs_unit = std_cogs * np.random.uniform(0.98, 1.02)

            total_cogs = round(units_sold * actual_cogs_unit)

            # Gross Profit
            gross_profit = net_revenue - total_cogs

            # Marketing A&P Spend
            ap_spend = round(net_revenue * np.random.uniform(0.04, 0.08))

            sales_rows.append(
                {
                    "Month": month_str,
                    "SKU_Code": sku,
                    "Channel": ch,
                    "Units_Sold": units_sold,
                    "Base_List_Price": base_price,
                    "Gross_Revenue": gross_revenue,
                    "Trade_Discount": trade_discount,
                    "Rebates_Listing_Fee": rebates_listing_fee,
                    "Net_Revenue": net_revenue,
                    "Total_COGS": total_cogs,
                    "Gross_Profit": gross_profit,
                    "Marketing_AP": ap_spend,
                }
            )

df_fact_sales = pd.DataFrame(sales_rows)

# 3. GENERATE FACT_WORKING_CAPITAL_MONTHLY
wc_rows = []

for m in months:
    month_str = m.strftime("%Y-%m")
    month_num = m.month

    # Filter total sales & COGS untuk bulan berjalan
    monthly_sales = df_fact_sales[df_fact_sales["Month"] == month_str]
    credit_sales = monthly_sales["Net_Revenue"].sum()
    cogs_total = monthly_sales["Total_COGS"].sum()

    # Base DSO ~40-45 hari, DPO ~50-60 hari, DIO ~45-50 hari
    dso_target = np.random.uniform(40, 46)
    dpo_target = np.random.uniform(52, 58)

    # ANOMALI 3: Working Capital Inefficiency / Penumpukan Stok di Q2 (Bulan 6)
    if month_num == 6:
        dio_target = np.random.uniform(70, 78)  # Inventory membengkak parah
    elif month_num in [7, 8]:
        dio_target = np.random.uniform(58, 65)  # Normalisasi bertahap
    else:
        dio_target = np.random.uniform(42, 48)

    # Menghitung Posisi Neraca berdasarkan target hari (30 hari / bulan)
    accounts_receivable = round((credit_sales / 30) * dso_target)
    inventory_value = round((cogs_total / 30) * dio_target)
    accounts_payable = round((cogs_total / 30) * dpo_target)

    # Metrik Turunan
    dso = round((accounts_receivable / credit_sales) * 30, 1) if credit_sales > 0 else 0
    dio = round((inventory_value / cogs_total) * 30, 1) if cogs_total > 0 else 0
    dpo = round((accounts_payable / cogs_total) * 30, 1) if cogs_total > 0 else 0
    ccc = round(dio + dso - dpo, 1)

    wc_rows.append(
        {
            "Month": month_str,
            "Credit_Sales": credit_sales,
            "Monthly_COGS": cogs_total,
            "Accounts_Receivable": accounts_receivable,
            "Inventory_Value": inventory_value,
            "Accounts_Payable": accounts_payable,
            "DSO_Days": dso,
            "DIO_Days": dio,
            "DPO_Days": dpo,
            "CCC_Days": ccc,
        }
    )

df_fact_working_capital = pd.DataFrame(wc_rows)

# 4. EKSPOR DATASET KE CSV
df_dim_product.to_csv("dim_product.csv", index=False)
df_fact_sales.to_csv("fact_sales_monthly.csv", index=False)
df_fact_working_capital.to_csv("fact_working_capital_monthly.csv", index=False)